# 01 -- Build the modelling base table

**What this notebook does (plain English):** This turns the assembled data into
the table every model will use. For each loan it records three things that drive
all later numbers:

- **Default** -- did the loan go badly wrong? (180+ days late, or it ended in a
  loss event such as a foreclosure sale.)
- **EAD (Exposure at Default)** -- how much money was still owed when it defaulted.
- **LGD (Loss Given Default)** -- of that exposure, how much was *actually lost*
  after the property was sold and costs/recoveries settled. This is computed from
  Freddie Mac's **real loss fields**, which is the centrepiece of the project.

**Headline result:** average loss-given-default is far worse in the downturn
(~55-58% in 2007/2008) than in the calm year (~25% in 2015).

In [1]:
import sys, os
ROOT = os.getcwd()
if not os.path.isdir(os.path.join(ROOT, 'src')):
    ROOT = os.path.dirname(ROOT)
os.chdir(ROOT)
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)
import warnings; warnings.filterwarnings('ignore')
print('project root:', ROOT)

project root: D:\Jane\Job Search\Github\bank\github project\freddie mac mortgage


In [2]:
# Load the cached loan-level table from notebook 00.
import numpy as np
import pandas as pd
from src import definitions as d
from src.output import save_csv
df = pd.read_parquet('data/processed/loan_level.parquet')
print(df.shape)

(150000, 55)


In [3]:
# Realised loss and LGD are only defined for defaulted loans that DISPOSED
# (reached a final sale). For everyone else LGD is left blank, on purpose.
df['realised_loss'] = np.where(df['disposed'], d.realised_loss(df), np.nan)
lgd_raw = df['realised_loss'] / df['ead'].replace(0, np.nan)
df['lgd'] = np.where(df['disposed'], d.winsorise_lgd(lgd_raw), np.nan)

In [4]:
# Sanity check: reconcile our computed loss against Freddie Mac's own
# actual_loss_calculation field (their number is stored as a negative loss).
rec = df[df['disposed']].dropna(subset=['actual_loss_calculation'])
rec = rec[rec['actual_loss_calculation'] != 0]
corr = np.corrcoef(-rec['actual_loss_calculation'], d.realised_loss(rec))[0, 1]
print(f'loss reconciliation correlation vs dataset field: {corr:.3f}')

loss reconciliation correlation vs dataset field: 0.990


In [5]:
# Add simple risk bands we will reuse in the EDA and models.
df['credit_score_band'] = pd.cut(df['credit_score'], [0, 620, 660, 700, 740, 780, 851],
                                 right=False, labels=['<620', '620-659', '660-699', '700-739', '740-779', '780+'])
df['ltv_band'] = pd.cut(df['original_ltv'], [0, 60, 70, 80, 90, 200],
                        right=False, labels=['<60', '60-69', '70-79', '80-89', '90+'])

### The workout period (input to the discounting step)

When a loan defaults the money is **not** lost all at once -- the lender works
through foreclosure and sale over many months, and a dollar recovered years later
is worth less than a dollar today. The **workout period** is how long that takes:
from the **default month** to the **disposition month**.

- The **default month** (`default_period`) already comes straight from the data --
  the first month the loan was 180+ days late or hit a loss event.
- The **disposition month** (`disposition_period`) uses the **exact zero-balance
  effective date** where Freddie Mac records it, and falls back automatically to
  the **last servicing month** (`last_period`) when that date is missing.

`months_to_resolution` is that gap in whole months, and it is only meaningful for
**disposed defaults** (NaN everywhere else). Together with `original_interest_rate`
it is one of the two inputs the economic-loss discounting in notebook 01 /
`definitions.economic_loss()` will use (see LGD alignment task P1-1).

In [6]:
# Disposition month: exact zero-balance date if loaded, else last servicing month.
if 'disposition_period' in df.columns:
    df['disposition_period'] = df['disposition_period'].fillna(df['last_period'])
else:
    df['disposition_period'] = df['last_period']

# Workout length in months, only meaningful for disposed defaults.
df['months_to_resolution'] = np.where(
    df['disposed'],
    d.months_between(df['default_period'], df['disposition_period']),
    np.nan,
)

In [7]:
# Keep one clean analysis row per loan and cache it for later notebooks.
base_cols = [
    'loan_sequence_number', 'vintage_year', 'credit_score', 'original_ltv',
    'original_cltv', 'original_dti', 'original_interest_rate', 'original_loan_term',
    'original_upb', 'loan_purpose', 'occupancy_status', 'channel', 'number_of_borrowers',
    'credit_score_band', 'ltv_band', 'ever_default', 'disposed', 'max_delinq_status',
    'ead', 'realised_loss', 'lgd',
    'default_period', 'disposition_period', 'months_to_resolution',
]
base = df[base_cols].copy()
base.to_parquet('data/processed/analysis_base.parquet')
print('analysis base:', base.shape)

analysis base: (150000, 24)


In [8]:
# Sanity-check the new workout-length field on disposed defaults only.
wr = df.loc[df['disposed'], 'months_to_resolution']
print('disposed defaults with a usable months_to_resolution:', int(wr.notna().sum()))
print('min / median / max months: {:.0f} / {:.0f} / {:.0f}'.format(
    wr.min(), wr.median(), wr.max()))
print('share resolved in 0 months:', round(float((wr == 0).mean()), 4))
# Confirm the field is blank for everyone who did NOT dispose-as-default.
print('non-disposed loans with a non-NaN value (should be 0):',
      int(df.loc[~df['disposed'], 'months_to_resolution'].notna().sum()))

disposed defaults with a usable months_to_resolution: 6749
min / median / max months: 0 / 14 / 180
share resolved in 0 months: 0.0455
non-disposed loans with a non-NaN value (should be 0): 0


In [9]:
# Results table: default rate and average LGD by vintage (downturn vs calm).
tbl = df.groupby('vintage_year').agg(
    loans=('loan_sequence_number', 'size'),
    default_rate=('ever_default', 'mean'),
    disposed_defaults=('disposed', 'sum'),
    avg_lgd=('lgd', 'mean'),
    median_lgd=('lgd', 'median'),
    avg_ead=('ead', 'mean'),
).reset_index().round(4)
save_csv(tbl, 'output/01_default_lgd_by_vintage.csv')
tbl

,vintage_year,loans,default_rate,disposed_defaults,avg_lgd,median_lgd,avg_ead
0,2007,50000,0.1374,4479,0.5783,0.5631,186177.0150
1,2008,50000,0.0735,2134,0.5441,0.5131,198213.9893
2,2015,50000,0.0242,136,0.2464,0.1678,197949.9096


**Reading the table:** both the chance of default *and* the severity of
loss when it happens are much worse in the crisis vintages -- the two effects
compound, which is exactly why a downturn hurts a mortgage book so much.